<center>

# Fund P&L Attribution Report

### Anagha Athreyas


---

## Overview

This project builds an automated monthly portfolio performance report, the kind of analysis a portfolio manager or fund analyst would review at the end of every month to understand what drove returns and where the risks are.

It goes beyond simply tracking whether stocks went up or down. The goal is to answer a more useful question: **which positions, sectors, and regions are actually driving performance, and which ones are dragging it down?**

---

## Motivation

Stock prices move every day, but raw price changes tell you very little on their own.

To make sense of performance, you need to attribute it: break it down by position, by sector, by geography, and compare it against a baseline. That is what this tool does. It takes a portfolio of 14 real stocks, calculates monthly P&L at the position level, and rolls it up into a structured management report automatically.

---

## Portfolio Construction

Stocks were selected using four criteria to ensure the portfolio is realistic and institutionally sound:

| Rule | Criteria | Rationale |
|------|----------|-----------|
| 1 | Market Cap above $100B | Large and mega-cap stocks only, more liquid, more stable, and more widely held by institutional investors |
| 2 | Average Daily Volume above $500M | Ensures positions can be entered and exited at scale without moving the market |
| 3 | Six sectors represented | Reduces concentration risk so a downturn in one industry does not drive outsized losses |
| 4 | Three regions represented | Geographic diversification protects against country-specific macro events |

Sectors: Technology, Financials, Healthcare, Energy, Consumer Discretionary, Consumer Staples

Regions: North America, Europe, Asia Pacific

---

## Currency Handling

All figures are reported in USD. Nestlé (NESN.SW) trades on the Swiss exchange in Swiss francs, so its prices are converted to USD using the daily CHF/USD exchange rate before any P&L or NAV is calculated. All other positions are US-listed and already priced in USD.

---

## Report Structure

Running the notebook generates a formatted Excel report with five sheets:

| Sheet | Content |
|-------|---------|
| Fund Summary | Top level KPIs: total NAV, actual P&L, vs budget, vs prior month |
| Positions | Security level breakdown with open price, close price, return and P&L |
| Sector | P&L attribution by sector with totals row |
| Geography | P&L attribution by region with totals row |
| Top Movers | Top 3 contributors and top 3 detractors for the month |

---

## Data

All prices are sourced live from Yahoo Finance via the yfinance Python library. No API key required. The report automatically pulls data for the most recently completed calendar month and the month prior for comparison.

---

In [ ]:
import os
print("Working directory:", os.getcwd())

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

PORTFOLIO = {
    "AAPL":    {"shares": 50,  "sector": "Technology",       "geo": "North America", "budget_return_pct": 2.0},
    "MSFT":    {"shares": 30,  "sector": "Technology",       "geo": "North America", "budget_return_pct": 1.8},
    "GOOGL":   {"shares": 15,  "sector": "Technology",       "geo": "North America", "budget_return_pct": 1.5},
    "JPM":     {"shares": 40,  "sector": "Financials",       "geo": "North America", "budget_return_pct": 1.2},
    "GS":      {"shares": 20,  "sector": "Financials",       "geo": "North America", "budget_return_pct": 1.0},
    "JNJ":     {"shares": 25,  "sector": "Healthcare",       "geo": "North America", "budget_return_pct": 0.8},
    "UNH":     {"shares": 18,  "sector": "Healthcare",       "geo": "North America", "budget_return_pct": 1.1},
    "XOM":     {"shares": 35,  "sector": "Energy",           "geo": "North America", "budget_return_pct": 0.5},
    "SHEL":    {"shares": 30,  "sector": "Energy",           "geo": "Europe",        "budget_return_pct": 0.6},
    "ASML":    {"shares": 12,  "sector": "Technology",       "geo": "Europe",        "budget_return_pct": 1.4},
    "NESN.SW": {"shares": 20,  "sector": "Consumer Staples", "geo": "Europe",        "budget_return_pct": 0.7},
    "TM":      {"shares": 22,  "sector": "Consumer Disc.",   "geo": "Asia Pacific",  "budget_return_pct": 0.9},
    "TSM":     {"shares": 28,  "sector": "Technology",       "geo": "Asia Pacific",  "budget_return_pct": 1.6},
    "BABA":    {"shares": 25,  "sector": "Consumer Disc.",   "geo": "Asia Pacific",  "budget_return_pct": -0.5},
}

CASH_BALANCE = 75_000.00
FUND_NAME = "Global Multi-Asset Fund"
REPORTING_CURRENCY = "USD"

# Non-USD listings and the FX pair used to convert them into USD
FX_MAP = {"NESN.SW": "CHFUSD=X"}

def get_month_boundaries(offset_months=0):
    today = datetime.today()
    target = today - relativedelta(months=offset_months)
    first = target.replace(day=1)
    last = (first + relativedelta(months=1)) - timedelta(days=1)
    return first, last

def fetch_monthly_prices(tickers, month_offset=0):
    first, last = get_month_boundaries(offset_months=month_offset)
    buffer_start = first - timedelta(days=5)
    buffer_end = last + timedelta(days=5)
    print(f"  Fetching {first.strftime('%b %Y')} prices...")
    fx_tickers = sorted(set(FX_MAP.values()))
    raw = yf.download(tickers + fx_tickers, start=buffer_start.strftime("%Y-%m-%d"),
                      end=buffer_end.strftime("%Y-%m-%d"), auto_adjust=True, progress=False)
    prices = raw["Close"]
    prices = prices[(prices.index >= pd.Timestamp(first)) & (prices.index <= pd.Timestamp(last))]
    prices = prices.dropna(how="all")
    result = {}
    for ticker in tickers:
        if ticker in prices.columns and not prices[ticker].dropna().empty:
            series = prices[ticker].dropna()
            open_px = float(series.iloc[0])
            close_px = float(series.iloc[-1])
            if ticker in FX_MAP:
                fx = prices[FX_MAP[ticker]].ffill().bfill()
                open_px *= float(fx.loc[series.index[0]])
                close_px *= float(fx.loc[series.index[-1]])
            result[ticker] = {"open": open_px, "close": close_px}
        else:
            result[ticker] = {"open": np.nan, "close": np.nan}
    return result

tickers = list(PORTFOLIO.keys())
print("Fetching market data...")
current_prices = fetch_monthly_prices(tickers, month_offset=1)
prior_prices   = fetch_monthly_prices(tickers, month_offset=2)

_, last_day = get_month_boundaries(offset_months=1)
month_label = last_day.strftime("%B %Y")

rows = []
for ticker, meta in PORTFOLIO.items():
    cur = current_prices.get(ticker, {})
    prev = prior_prices.get(ticker, {})
    cur_open    = cur.get("open",   np.nan)
    cur_close   = cur.get("close",  np.nan)
    prior_open  = prev.get("open",  np.nan)
    prior_close = prev.get("close", np.nan)
    shares = meta["shares"]
    ok_cur   = not (np.isnan(cur_open)   or np.isnan(cur_close))
    ok_prior = not (np.isnan(prior_open) or np.isnan(prior_close))
    cur_pnl        = round(shares * (cur_close - cur_open), 2)                     if ok_cur   else None
    cur_return_pct = round((cur_close - cur_open) / cur_open * 100, 2)             if ok_cur   else None
    prior_pnl      = round(shares * (prior_close - prior_open), 2)                 if ok_prior else None
    budget_pnl     = round(shares * cur_open * (meta["budget_return_pct"]/100), 2) if ok_cur   else None
    vs_budget      = round(cur_pnl - budget_pnl, 2) if (cur_pnl is not None and budget_pnl is not None) else None
    vs_prior       = round(cur_pnl - prior_pnl,  2) if (cur_pnl is not None and prior_pnl  is not None) else None
    rows.append({
        "Ticker":              ticker,
        "Sector":              meta["sector"],
        "Geography":           meta["geo"],
        "Shares":              shares,
        "Month Open ($)":      round(cur_open,  2) if ok_cur else None,
        "Month Close ($)":     round(cur_close, 2) if ok_cur else None,
        "Monthly Return (%)":  cur_return_pct,
        "Actual P&L ($)":      cur_pnl,
        "Prior Month P&L ($)": prior_pnl,
        "Budget P&L ($)":      budget_pnl,
        "Vs. Budget ($)":      vs_budget,
        "Vs. Prior Month ($)": vs_prior,
    })

position_df = pd.DataFrame(rows)
print("\nData fetched. Preview:")
print(position_df[["Ticker", "Sector", "Geography", "Monthly Return (%)", "Actual P&L ($)"]].to_string(index=False))

In [ ]:
def build_sector_attribution(df):
    return (df.groupby("Sector").agg(**{
        "Actual P&L ($)":      ("Actual P&L ($)",      "sum"),
        "Prior Month P&L ($)": ("Prior Month P&L ($)", "sum"),
        "Budget P&L ($)":      ("Budget P&L ($)",      "sum"),
        "Vs. Budget ($)":      ("Vs. Budget ($)",      "sum"),
        "Vs. Prior Month ($)": ("Vs. Prior Month ($)", "sum"),
    }).round(2).sort_values("Actual P&L ($)", ascending=False).reset_index())

def build_geo_attribution(df):
    return (df.groupby("Geography").agg(**{
        "Actual P&L ($)":      ("Actual P&L ($)",      "sum"),
        "Prior Month P&L ($)": ("Prior Month P&L ($)", "sum"),
        "Budget P&L ($)":      ("Budget P&L ($)",      "sum"),
        "Vs. Budget ($)":      ("Vs. Budget ($)",      "sum"),
        "Vs. Prior Month ($)": ("Vs. Prior Month ($)", "sum"),
    }).round(2).sort_values("Actual P&L ($)", ascending=False).reset_index())

sector_df = build_sector_attribution(position_df)
geo_df    = build_geo_attribution(position_df)

s = position_df.dropna(subset=["Actual P&L ($)"]).sort_values("Actual P&L ($)", ascending=False)
cols = ["Ticker", "Sector", "Geography", "Actual P&L ($)", "Monthly Return (%)"]
contributors = s.head(3)[cols].reset_index(drop=True)
detractors   = s.tail(3).iloc[::-1][cols].reset_index(drop=True)

total_mv = position_df["Shares"].multiply(position_df["Month Close ($)"].fillna(0)).sum() + CASH_BALANCE
summary = {
    "Fund":                   FUND_NAME,
    "Reporting Month":        month_label,
    "Reporting Currency":     REPORTING_CURRENCY,
    "Total Fund NAV ($)":     round(total_mv, 2),
    "Total Actual P&L ($)":   round(position_df["Actual P&L ($)"].sum(), 2),
    "Total Budget P&L ($)":   round(position_df["Budget P&L ($)"].sum(), 2),
    "Total Prior P&L ($)":    round(position_df["Prior Month P&L ($)"].sum(), 2),
    "Vs. Budget ($)":         round(position_df["Actual P&L ($)"].sum() - position_df["Budget P&L ($)"].sum(), 2),
    "Vs. Prior Month ($)":    round(position_df["Actual P&L ($)"].sum() - position_df["Prior Month P&L ($)"].sum(), 2),
}

print(f"Fund Summary: {month_label}")
print(f"  Total NAV:       ${summary['Total Fund NAV ($)']:>12,.2f}")
print(f"  Actual P&L:      ${summary['Total Actual P&L ($)']:>12,.2f}")
print(f"  Vs. Budget:      ${summary['Vs. Budget ($)']:>+12,.2f}")
print(f"  Vs. Prior Month: ${summary['Vs. Prior Month ($)']:>+12,.2f}")
print("\nTop 3 Contributors:")
print(contributors[["Ticker", "Actual P&L ($)", "Monthly Return (%)"]].to_string(index=False))
print("\nTop 3 Detractors:")
print(detractors[["Ticker", "Actual P&L ($)", "Monthly Return (%)"]].to_string(index=False))

In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment

os.makedirs("output", exist_ok=True)
file_name = f"output/PnL_Attribution_{last_day.strftime('%b_%Y')}_{datetime.today().strftime('%Y%m%d')}.xlsx"

summary_df = pd.DataFrame(list(summary.items()), columns=["Metric", "Value"])

def add_totals(df, label_col):
    totals = df.select_dtypes("number").sum().round(2)
    totals[label_col] = "Total"
    return pd.concat([df, pd.DataFrame([totals])], ignore_index=True)

sector_out = add_totals(sector_df, "Sector")
geo_out    = add_totals(geo_df, "Geography")
movers = pd.concat([contributors.assign(Type="Contributor"),
                    detractors.assign(Type="Detractor")], ignore_index=True)

header_fill = PatternFill("solid", fgColor="1F3864")
green_fill  = PatternFill("solid", fgColor="C6EFCE")
red_fill    = PatternFill("solid", fgColor="FFC7CE")

with pd.ExcelWriter(file_name, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="Fund Summary", index=False)
    position_df.to_excel(writer, sheet_name="Positions", index=False)
    sector_out.to_excel(writer, sheet_name="Sector", index=False)
    geo_out.to_excel(writer, sheet_name="Geography", index=False)
    movers.to_excel(writer, sheet_name="Top Movers", index=False)

    for ws in writer.book.worksheets:
        for cell in ws[1]:
            cell.font = Font(bold=True, color="FFFFFF")
            cell.fill = header_fill
            cell.alignment = Alignment(horizontal="center")
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                if isinstance(cell.value, (int, float)):
                    cell.number_format = "#,##0.00"
        for col in ws.columns:
            width = max(len(str(c.value)) if c.value is not None else 0 for c in col)
            ws.column_dimensions[col[0].column_letter].width = width + 3

    for name in ("Sector", "Geography"):
        ws = writer.book[name]
        for cell in ws[ws.max_row]:
            cell.font = Font(bold=True)

    ws = writer.book["Top Movers"]
    type_col = ws.max_column
    for r in range(2, ws.max_row + 1):
        fill = green_fill if ws.cell(row=r, column=type_col).value == "Contributor" else red_fill
        for c in range(1, ws.max_column + 1):
            ws.cell(row=r, column=c).fill = fill

print(f"Report saved to: {file_name}")